In [1]:
import os

# Limit native linear-algebra thread pools before NumPy/SciPy are imported.
# This prevents VS Code/Jupyter kernels from crashing on Windows with OpenBLAS NUM_THREADS errors.
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ast
from scipy import stats
import librosa
import warnings
import pandas as pd

In [2]:
paths = {}
print(os.path.join(os.getcwd(), "fma_small"))
curr_path = os.path.join(os.getcwd(), "fma_small")
for root, dirs, files in os.walk(curr_path):
    if root.endswith("fma_small"):
        continue
    for file in files:
        full_path = os.path.join(root, file)
        track_id = int(file.split(".")[0])
        paths[track_id] = full_path

c:\Users\Jakub Karczewski\data_exploration_project\fma_small


In [3]:
paths
len(list(paths.keys()))

8000

In [4]:
def columns():
    feature_sizes = dict(
        chroma_stft=12,
        mfcc=20, rms=1, zcr=1,
        spectral_centroid=1
    )
    moments = ('mean', 'std')

    tuples = [
        (feature, moment, f"{i+1:02d}")
        for feature, size in feature_sizes.items()
        for moment in moments
        for i in range(size)
    ]

    return pd.MultiIndex.from_tuples(tuples, names=('feature', 'statistics', 'number')).sort_values()

In [5]:
def compute_features_librosa(tid):

    features = pd.Series(index=columns(), dtype=np.float32, name=tid)

    # Catch warnings as exceptions (audioread leaks file descriptors).
    warnings.filterwarnings('error', module='librosa')

    def feature_stats(name, values):
        # values.shape = (num_rows, num_frames)
        for i in range(values.shape[0]):
            features[name, 'mean', f'{i+1:02d}'] = np.float32(np.mean(values[i, :]))
            features[name, 'std', f'{i+1:02d}'] = np.float32(np.std(values[i, :]))
            #features[name, 'skew', f'{i+1:02d}'] = np.float32(stats.skew(values[i, :]))
            #features[name, 'kurtosis', f'{i+1:02d}'] = np.float32(stats.kurtosis(values[i, :]))
            #features[name, 'median', f'{i+1:02d}'] = np.float32(np.median(values[i, :]))
            #features[name, 'min', f'{i+1:02d}'] = np.float32(np.min(values[i, :]))
            #features[name, 'max', f'{i+1:02d}'] = np.float32(np.max(values[i, :]))

    try:
        filepath = paths[tid]
        x, sr = librosa.load(filepath, sr=22050, mono=True)

        f = librosa.feature.zero_crossing_rate(x, frame_length=2048, hop_length=512)
        feature_stats('zcr', f)

        stft = np.abs(librosa.stft(x, n_fft=2048, hop_length=512))
        del x

        f = librosa.feature.chroma_stft(S=stft**2, n_chroma=12)
        feature_stats('chroma_stft', f)

        f = librosa.feature.rms(S=stft)
        feature_stats('rms', f)

        f = librosa.feature.spectral_centroid(S=stft)
        feature_stats('spectral_centroid', f)

        mel = librosa.feature.melspectrogram(sr=sr, S=stft**2)
        del stft
        f = librosa.feature.mfcc(S=librosa.power_to_db(mel), n_mfcc=20)
        feature_stats('mfcc', f)

    except Exception as e:
        print('{}: {}'.format(tid, repr(e)))
        
    return features




In [6]:
# !pip install essentia

In [7]:
from multiprocessing.pool import ThreadPool
from tqdm import tqdm

track_ids = list(paths.keys())

features = pd.DataFrame(
    index=track_ids,
    columns=columns(),
    dtype=np.float32
)

nb_workers = int(1.5 * os.cpu_count())

pool = ThreadPool(nb_workers)

for row in tqdm(pool.imap_unordered(compute_features_librosa, track_ids), total=len(track_ids)):
    features.loc[row.name] = row

pool.close()
pool.join()


  0%|          | 0/8000 [00:00<?, ?it/s]C:\Users\Jakub Karczewski\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
 20%|█▉        | 1598/8000 [01:19<03:06, 34.24it/s]

41381: UserWarning('Trying to estimate tuning from empty frequency set.')


 38%|███▊      | 3015/8000 [02:24<02:46, 29.91it/s]

69002: UserWarning('Trying to estimate tuning from empty frequency set.')


 55%|█████▍    | 4388/8000 [03:23<04:01, 14.94it/s]C:\Users\Jakub Karczewski\AppData\Local\Temp\ipykernel_28612\2647347381.py:21: UserWarning: PySoundFile failed. Trying audioread instead.
  x, sr = librosa.load(filepath, sr=22050, mono=True)
C:\Users\Jakub Karczewski\AppData\Local\Temp\ipykernel_28612\2647347381.py:21: UserWarning: PySoundFile failed. Trying audioread instead.
  x, sr = librosa.load(filepath, sr=22050, mono=True)
C:\Users\Jakub Karczewski\AppData\Local\Temp\ipykernel_28612\2647347381.py:21: UserWarning: PySoundFile failed. Trying audioread instead.
  x, sr = librosa.load(filepath, sr=22050, mono=True)
 55%|█████▍    | 4394/8000 [03:23<03:08, 19.11it/s]

...

 76%|███████▌  | 6082/8000 [04:36<01:09, 27.72it/s]

123502: UserWarning('Trying to estimate tuning from empty frequency set.')


 87%|████████▋ | 6929/8000 [05:24<01:01, 17.39it/s]C:\Users\Jakub Karczewski\AppData\Local\Temp\ipykernel_28612\2647347381.py:21: UserWarning: PySoundFile failed. Trying audioread instead.
  x, sr = librosa.load(filepath, sr=22050, mono=True)
 87%|████████▋ | 6931/8000 [05:24<01:05, 16.27it/s]

133297: FutureWarning('librosa.core.audio.__audioread_load\n\tDeprecated as of librosa version 0.10.0.\n\tIt will be removed in librosa version 1.0.')


100%|██████████| 8000/8000 [06:28<00:00, 20.62it/s]


In [8]:
features.to_csv(
    "features_small_librosa.csv",
    float_format="%.6f",
    header=['_'.join(col).strip() for col in features.columns.values],
    index_label="track_id"
)
features

feature    chroma_stft                                                    \
statistics        mean                                                     
number              01        02        03        04        05        06   
2             0.747006  0.465571  0.376991  0.346401  0.276217  0.234040   
5             0.408774  0.578310  0.398189  0.341066  0.322690  0.329029   
10            0.283685  0.649419  0.330624  0.347292  0.231403  0.323438   
140           0.242838  0.270964  0.364772  0.345893  0.379862  0.433202   
141           0.164104  0.153202  0.243002  0.186743  0.234106  0.373787   
...                ...       ...       ...       ...       ...       ...   
154308        0.342092  0.652999  0.303526  0.277715  0.358287  0.235484   
154309        0.244645  0.213864  0.431622  0.305290  0.478040  0.262319   
154413        0.555321  0.393367  0.434515  0.357703  0.451547  0.393143   
154414        0.240129  0.272328  0.393441  0.307411  0.557563  0.283900   
155066        0.362122  0.499583  0.346401  0.430137  0.496885  0.538016   

feature                                             ...       mfcc             \
statistics                                          ...        std              
number            07        08        09        10  ...         17         18   
2           0.272904  0.419900  0.372483  0.358381  ...   7.308831   7.230115   
5           0.373279  0.398300  0.488417  0.491822  ...   6.713108   6.629267   
10          0.700743  0.338985  0.381055  0.308978  ...   5.731713   6.350440   
140         0.273431  0.234015  0.200707  0.274969  ...   6.154854   6.254518   
141         0.242960  0.225383  0.214111  0.385499  ...   9.598772   7.808608   
...              ...       ...       ...       ...  ...        ...        ...   
154308      0.307607  0.307700  0.717316  0.357831  ...   9.425747  10.397196   
154309      0.430785  0.386952  0.214757  0.202658  ...   8.127104  10.184349   
154413      0.362939  0.361938  0.253395  0.353543  ...  12.346698  15.994142   
154414      0.228731  0.261790  0.279456  0.303360  ...   7.354480   8.261937   
155066      0.465652  0.386845  0.630165  0.569591  ...   5.696255   5.314806   

feature                                rms           spectral_centroid  \
statistics                            mean       std              mean   
number             19         20        01        01                01   
2            7.195039   6.366723  0.085953  0.053469       3056.587891   
5            8.085360   8.035872  0.088805  0.053615       2430.188965   
10           5.508020   5.445374  0.114076  0.031983       2358.129150   
140          6.819814   6.646987  0.041877  0.024219       1765.877808   
141          8.252582   9.453534  0.061718  0.048319       1666.702881   
...               ...        ...       ...       ...               ...   
154308       8.337333   8.049065  0.040824  0.038814       1698.633301   
154309      13.032768  12.925120  0.041588  0.043767       2600.636719   
154413      14.335363  14.541687  0.097349  0.016937       1346.634277   
154414       8.717627   9.326102  0.077728  0.022157       2186.769287   
155066       5.675515   5.776505  0.154966  0.019260        494.036469   

feature                       zcr            
statistics          std      mean       std  
number               01        01        01  
2            997.470154  0.164406  0.093938  
5            878.139465  0.100544  0.067248  
10           382.231567  0.148947  0.028327  
140         1036.000610  0.044525  0.052389  
141          809.798645  0.062031  0.052299  
...                 ...       ...       ...  
154308       770.849426  0.076531  0.054476  
154309      1789.580322  0.112923  0.089936  
154413       826.892395  0.038495  0.030414  
154414       925.760864  0.090261  0.056155  
155066       322.426666  0.017363  0.007670  

[8000 rows x 70 columns]